In [ ]:
# Author: Arun Bamal
# Project: Data Engineering Assignmentimport duckdb
import pandas as pd
import time

print("Libraries loaded")

Libraries loaded


In [3]:
import duckdb
import pandas as pd
import time

con = duckdb.connect("ecommerce.db")

print("Connected")

Connected


In [4]:
start = time.time()

duckdb_result = con.execute("""
SELECT 
    event_type,
    COUNT(*) as total_events,
    SUM(price) as total_revenue
FROM clean_events
GROUP BY event_type
""").fetchdf()

duckdb_time = time.time() - start

print("DuckDB Time:", duckdb_time)
duckdb_result

DuckDB Time: 0.23855948448181152


,event_type,total_events,total_revenue
0,cart,926516,3.092988e+08
1,purchase,742849,2.299575e+08
2,view,40779399,1.178462e+10


In [6]:
start = time.time()

df = pd.read_csv("C:/Users/arunb/Downloads/2019-Oct.csv")

pandas_load_time = time.time() - start

print("Pandas Load Time:", pandas_load_time)

Pandas Load Time: 70.20474696159363


In [7]:
start = time.time()

df = df.dropna(subset=["price"])
df["price"] = df["price"].astype(float)

pandas_transform_time = time.time() - start

print("Pandas Transform Time:", pandas_transform_time)

Pandas Transform Time: 0.10661959648132324


In [8]:
start = time.time()

pandas_result = df.groupby("event_type").agg({
    "price": ["count", "sum"]
})

pandas_query_time = time.time() - start

print("Pandas Query Time:", pandas_query_time)
pandas_result

Pandas Query Time: 3.8638932704925537


price              
               count           sum
event_type                        
cart          926516  3.092988e+08
purchase      742849  2.299575e+08
view        40779399  1.178462e+10

In [9]:
print("---- PERFORMANCE COMPARISON ----")

print(f"DuckDB Query Time: {duckdb_time:.4f} sec")
print(f"Pandas Load Time: {pandas_load_time:.4f} sec")
print(f"Pandas Transform Time: {pandas_transform_time:.4f} sec")
print(f"Pandas Query Time: {pandas_query_time:.4f} sec")

---- PERFORMANCE COMPARISON ----
DuckDB Query Time: 0.2386 sec
Pandas Load Time: 70.2047 sec
Pandas Transform Time: 0.1066 sec
Pandas Query Time: 3.8639 sec


In [10]:
comparison = pd.DataFrame({
    "Method": ["DuckDB Query", "Pandas Load", "Pandas Transform", "Pandas Query"],
    "Time (seconds)": [
        duckdb_time,
        pandas_load_time,
        pandas_transform_time,
        pandas_query_time
    ]
})

comparison

,Method,Time (seconds)
0,DuckDB Query,0.238559
1,Pandas Load,70.204747
2,Pandas Transform,0.106620
3,Pandas Query,3.863893


In [1]:
import duckdb
import time

start = time.time()

duckdb.query("""
SELECT category_code, SUM(price)
FROM read_csv_auto('C:/Users/arunb/Downloads/2019-Oct.csv')
GROUP BY category_code
""").df()

csv_time = time.time() - start
print("CSV Time:", csv_time)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV Time: 8.100223779678345


In [1]:
import duckdb
import time

con = duckdb.connect("ecommerce.db", read_only=True)

print("Benchmark started")

Benchmark started


In [2]:
start = time.time()

result = con.execute("""
SELECT 
    c.category_code,
    SUM(f.price) AS revenue
FROM fact_events f
JOIN dim_product p ON f.product_id = p.product_id
JOIN dim_category c ON p.category_id = c.category_id
GROUP BY c.category_code
ORDER BY revenue DESC
LIMIT 10;
""").fetchdf()

end = time.time()

print(result)
print("Query 1 Time:", round(end - start, 4), "seconds")

                      category_code       revenue
0            electronics.smartphone  5.430804e+09
1                computers.notebook  8.092572e+08
2              electronics.video.tv  4.921630e+08
3                electronics.clocks  3.860283e+08
4  appliances.kitchen.refrigerators  3.524488e+08
5         appliances.kitchen.washer  2.878749e+08
6                 computers.desktop  2.272596e+08
7     appliances.environment.vacuum  1.356794e+08
8        furniture.living_room.sofa  1.341838e+08
9                electronics.tablet  1.081085e+08
Query 1 Time: 1.4308 seconds


In [3]:
start = time.time()

result = con.execute("""
SELECT 
    user_id,
    COUNT(*) AS total_events
FROM fact_events
GROUP BY user_id
ORDER BY total_events DESC
LIMIT 10;
""").fetchdf()

end = time.time()

print(result)
print("Query 2 Time:", round(end - start, 4), "seconds")

     user_id  total_events
0  512475445          6703
1  512365995          3986
2  512505687          2886
3  513021392          2860
4  514649263          2377
5  546270188          2201
6  526731152          2148
7  551211823          2120
8  563459593          1950
9  512792872          1935
Query 2 Time: 0.3288 seconds


In [4]:
start = time.time()

result = con.execute("""
SELECT 
    d.hour,
    COUNT(*) AS total_events
FROM fact_events f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.hour
ORDER BY d.hour;
""").fetchdf()

end = time.time()

print(result)
print("Query 3 Time:", round(end - start, 4), "seconds")


    hour  total_events
0      0        196719
1      1        361473
2      2        700416
3      3       1038599
4      4       1305010
5      5       1459259
6      6       1560431
7      7       1609121
8      8       1644359
9      9       1620790
10    10       1581061
11    11       1514165
12    12       1482917
13    13       1630909
14    14       1843355
15    15       2037455
16    16       2066419
17    17       1836935
18    18       1412070
19    19        898453
20    20        503434
21    21        297419
22    22        186430
23    23        145954
Query 3 Time: 0.0432 seconds


In [5]:
start = time.time()

result = con.execute("""
SELECT 
    event_type,
    COUNT(*) AS total
FROM fact_events
GROUP BY event_type;
""").fetchdf()

end = time.time()

print(result)
print("Query 4 Time:", round(end - start, 4), "seconds")

  event_type     total
0       view  27542941
1   purchase    569424
2       cart    820788
Query 4 Time: 0.0377 seconds
